# Federated Learning Client 2
## Distributed Training Node for FL Simulation

This notebook implements **FL Client 2**, one of the distributed training nodes in our Federated Learning simulation.

---

### Client Responsibilities

| Role | Description |
|------|-------------|
| **Download Global Model** | Fetch the latest model from the FL server |
| **Local Training** | Train the model on local private data |
| **Upload Weights** | Send trained weights back to server for aggregation |
| **Wait for Aggregation** | Sync with other clients between rounds |

---

### Prerequisites

1. **Run `server.ipynb` FIRST** and wait until the server is running
2. **Copy the ngrok public URL** from the server notebook
3. **Paste the URL** in the configuration cell below
4. Run all cells sequentially

# 1. Environment Setup

Set up the computing environment with required dependencies for local training.

### 1.1 Install Dependencies
Install PyTorch, EfficientNet, and HTTP client libraries.

In [ ]:
# Install dependencies
!pip install efficientnet_pytorch requests -q

  Preparing metadata (setup.py) ... done


### 1.2 Import Libraries
Import all necessary libraries for training and server communication.

In [ ]:
import os
import io
import gzip
import base64
import hashlib
import random
import requests
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from efficientnet_pytorch import EfficientNet
import time
print('Libraries imported!')

Libraries imported!


# 2. Data Setup

Download the local training dataset for this client. In a real FL scenario, each client would have its own private data stored locally.

### 2.1 Download Client Dataset
Clone the client-specific training data from the workshop repository.

In [ ]:
# ========================================
# DOWNLOAD DATASET FROM GITHUB
# ========================================

# GitHub repository
GITHUB_REPO_URL = 'https://github.com/YARSIAICenter/faradisa-workshop-2026'
DATA_FOLDER = 'day-3/data/client_2'  # Data untuk Client 2
LOCAL_DIR = 'workshop-data'

# Download only folder data
if not os.path.exists(LOCAL_DIR):
    print('Downloading Client 2 dataset...')
    !git clone --filter=blob:none --sparse {GITHUB_REPO_URL} {LOCAL_DIR}
    %cd {LOCAL_DIR}
    !git sparse-checkout set {DATA_FOLDER}
    %cd ..
    print('Download complete!')
else:
    print(f'Data available in folder {LOCAL_DIR}/')

# Set BASE_DIR to folder that consisting the data
BASE_DIR = os.path.join(LOCAL_DIR, 'day-3')
print(f'Data directory: {BASE_DIR}/')
print(f'Client 2 data: {os.path.join(BASE_DIR, "data/client_2")}')

Cloning into 'workshop-data'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 43 (delta 5), reused 34 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 42.09 KiB | 879.00 KiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/workshop-data
remote: Enumerating objects: 701, done.
remote: Counting objects: 100% (701/701), done.
remote: Compressing objects: 100% (701/701), done.
remote: Total 701 (delta 0), reused 701 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (701/701), 277.45 MiB | 19.33 MiB/s, done.
Updating files: 100% (701/701), done.
/content
Download complete!
Data directory: workshop-data/day-3/
Client 2 data: workshop-data/day-3/data/client_2


# 3. Server Connection

Configure the connection to the FL aggregation server. The server URL must be obtained from the running `server.ipynb` notebook.

### 3.1 Server URL Configuration

**⚠️ IMPORTANT: Paste the ngrok URL from the server notebook here!**

1. Make sure `server.ipynb` is running
2. Copy the public URL (e.g., `https://xxxx.ngrok-free.app`)
3. Paste it in the cell below

In [ ]:
# ========================================
# PASTE THE SERVER URL FROM SERVER NOTEBOOK
# ========================================

SERVER_URL = '...'  # Example: 'https://xxxx-xx-xx-xxx-xx.ngrok-free.dev'

# Validasi URL
if 'PASTE' in SERVER_URL or 'ngrok' not in SERVER_URL:
    print('='*60)
    print('WARNING: SERVER_URL is empty!')
    print('1. Run server.ipynb')
    print('2. Copy the ngrok URL')
    print('3. Paste the URL to the SERVER_URL variable')
    print('='*60)
else:
    print(f'Server URL: {SERVER_URL}')

Server URL: https://lily-snowplow-moodiness.ngrok-free.dev


### 3.2 Client and Training Configuration
Set up client identity, training hyperparameters, and reproducibility settings.

In [ ]:
# Client config
CLIENT_ID = '...' # Insert your client 2 identity
DATA_DIR = os.path.join(BASE_DIR, 'data/client_2')

# Training config
SEED = 42
N_CLASSES = 2
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 5e-4
LOCAL_EPOCHS = 1
MAX_ROUNDS = 3 # Insert the number of round (must be the same across all clients)
TRAIN_RATIO = 0.85

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Client ID: {CLIENT_ID}')
print(f'Data: {DATA_DIR}')
print(f'Server: {SERVER_URL}')
print(f'Device: {device}')

Client ID: PathGen
Data: workshop-data/day-3/data/client_2
Server: https://lily-snowplow-moodiness.ngrok-free.dev
Device: cpu


# 4. Model Architecture

Define the neural network architecture. This must be **identical** to the server's model to ensure weight compatibility.

**EfficientNet-B0 Model Definition**
Define the same model architecture as the server for weight compatibility.

In [ ]:
class EfficientNetB0(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.model = EfficientNet.from_pretrained('efficientnet-b0')
        self.num_ftrs = self.model._fc.in_features
        self.model._fc = nn.Linear(self.num_ftrs, n_classes)
        self.projector = nn.Sequential(
            nn.Linear(self.num_ftrs, self.num_ftrs),
            nn.Linear(self.num_ftrs, 1024)
        )

    def forward(self, x, project=False):
        features = self.model.extract_features(x)
        features = self.model._avg_pooling(features)
        features = features.flatten(start_dim=1)
        out = self.model._dropout(features)
        out = self.model._fc(out)
        return features, out

print('Model architecture defined!')

Model architecture defined!


# 5. Data Loading

Set up the local dataset and data loaders for training. The data is split into training and validation sets.

### 5.1 Dataset Class and DataLoaders
Define custom dataset class with data augmentation and create train/validation loaders.

In [ ]:
class LocalDataset(Dataset):
    def __init__(self, data_dir, transform, mode='all', seed=42):
        self.transform = transform
        samples = []
        csv_path = os.path.join(data_dir, 'labels.csv')
        images_dir = os.path.join(data_dir, 'images')
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            img_path = os.path.join(images_dir, row['filename'])
            if os.path.exists(img_path):
                samples.append((img_path, int(row['label'])))

        # Train/val split
        if mode in ['train', 'val']:
            np.random.seed(seed)
            indices = np.random.permutation(len(samples))
            split = int(0.85 * len(samples))
            if mode == 'train':
                indices = indices[:split]
            else:
                indices = indices[split:]
            samples = [samples[i] for i in indices]

        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        return self.transform(image), label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create dataloaders
train_dataset = LocalDataset(DATA_DIR, train_transform, mode='train', seed=SEED)
val_dataset = LocalDataset(DATA_DIR, val_transform, mode='val', seed=SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

DATA_SIZE = len(train_dataset)
print(f'Train samples: {DATA_SIZE}')
print(f'Val samples: {len(val_dataset)}')

Train samples: 595
Val samples: 105


# 6. FL Client Communication

Implement the communication protocol between client and server:
- Registration with the server
- Downloading the global model
- Uploading trained weights
- Waiting for aggregation completion

### 6.1 Server Communication Functions
Define functions for registering, downloading models, uploading weights, and synchronization.

In [ ]:
def register():
    '''Register client to server'''
    response = requests.post(f'{SERVER_URL}/register', json={
        'client_id': CLIENT_ID,
        'data_size': DATA_SIZE
    })
    result = response.json()
    print(f'Registered! Current round: {result["current_round"]}')
    return result['current_round']

def download_model():
    '''Download global model from server'''
    response = requests.get(f'{SERVER_URL}/model/download', params={'client_id': CLIENT_ID})
    if response.status_code != 200:
        raise Exception('Failed to download model')
    weights = torch.load(io.BytesIO(response.content), map_location=device)
    print('Global model downloaded!')
    return weights

def upload_weights(model, current_round):
    '''Upload weights using chunked upload with retry'''
    print('Uploading weights...')

    # Serialize and compress
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    weights_data = buffer.getvalue()
    compressed = gzip.compress(weights_data, compresslevel=9)
    encoded = base64.b64encode(compressed).decode('utf-8')

    print(f'  Original: {len(weights_data)/1024/1024:.2f} MB')
    print(f'  Compressed: {len(compressed)/1024/1024:.2f} MB')

    # Larger chunk size to reduce requests (2MB instead of 500KB)
    CHUNK_SIZE = 2 * 1024 * 1024
    total_chunks = (len(encoded) + CHUNK_SIZE - 1) // CHUNK_SIZE
    upload_id = hashlib.md5(f'{CLIENT_ID}_{current_round}_{datetime.now()}'.encode()).hexdigest()[:16]

    print(f'  Uploading {total_chunks} chunks...')

    for i in range(total_chunks):
        chunk = encoded[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE]
        # Retry mechanism for unstable connections
        for attempt in range(3):
            try:
                resp = requests.post(f'{SERVER_URL}/model/upload_chunk', json={
                    'client_id': CLIENT_ID,
                    'upload_id': upload_id,
                    'chunk_idx': i,
                    'total_chunks': total_chunks,
                    'chunk_data': chunk,
                    'data_size': DATA_SIZE,
                    'round': current_round
                }, timeout=60)
                if resp.status_code == 200:
                    break
            except Exception as e:
                if attempt < 2:
                    print(f'  Retry chunk {i+1}...')
                    time.sleep(2)
                else:
                    raise e
        print(f'  Chunk {i+1}/{total_chunks} uploaded')

    # Complete upload
    response = requests.post(f'{SERVER_URL}/model/upload_complete', json={
        'client_id': CLIENT_ID,
        'upload_id': upload_id,
        'data_size': DATA_SIZE,
        'round': current_round
    }, timeout=60)
    print('  Upload complete!')
    return response.json()

def wait_for_aggregation(current_round):
    '''Wait for server to complete aggregation'''
    print('Waiting for aggregation...')
    while True:
        try:
            response = requests.get(f'{SERVER_URL}/status', timeout=30)
            status = response.json()
            if status['current_round'] > current_round:
                print(f'Aggregation complete! Round {current_round} -> {status["current_round"]}')
                return status['current_round']
            print(f'  Waiting... ({status["weights_received_count"]}/{status["expected_clients"]} clients)')
        except:
            print('  Connection error, retrying...')
        time.sleep(5)

print('FL client functions defined!')

FL client functions defined!


# 7. Local Training

Implement the local training loop that runs on this client's private data.

### 7.1 Local Training Function
Define the training loop with optimizer, loss function, and progress tracking.

In [ ]:
def local_train(model, epochs=LOCAL_EPOCHS):
    '''Perform local training'''
    print(f'Starting local training ({epochs} epochs)...')
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            _, logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            if batch_idx % 10 == 0:
                print(f'  Batch {batch_idx}, Loss: {loss.item():.4f}')

        acc = 100. * correct / total
        avg_loss = running_loss / len(train_loader)
        print(f'Epoch {epoch+1}: Loss={avg_loss:.4f}, Acc={acc:.2f}%')

    print('Local training completed!')
    return model

print('Local training function defined!')

Local training function defined!


# 8. Run FL Client

Execute the federated learning client loop:
1. Check server connection
2. Register with server
3. For each round: download model → local training → upload weights → wait for aggregation

> **⚠️ Make sure `server.ipynb` is already running before executing these cells!**

### 8.1 Verify Server Connection
Test the connection to the FL server before starting training.

In [ ]:
# Check server connection
try:
    response = requests.get(f'{SERVER_URL}/health', timeout=5)
    print(f'Server connected! Status: {response.json()}')
except:
    print('ERROR: Server not running!')
    print('Please run server.ipynb')
    raise Exception('Server not available')

Server connected! Status: {'round': 0, 'status': 'healthy'}


### 8.2 Start Federated Learning
Run the main FL client loop for multiple rounds.

In [ ]:
# Register and start FL
current_round = register()
rounds_completed = 0

print('='*50)
print(f'FL CLIENT {CLIENT_ID} STARTING')
print(f'Max rounds: {MAX_ROUNDS}')
print('='*50)

while rounds_completed < MAX_ROUNDS:
    print(f'\n--- Round {current_round} ---')

    # 1. Download global model
    global_weights = download_model()
    model = EfficientNetB0(N_CLASSES).to(device)
    model.load_state_dict(global_weights)

    # 2. Local training
    model = local_train(model)

    # 3. Upload weights
    result = upload_weights(model, current_round)
    print(f'Upload result: {result["status"]}')

    # 4. Wait for aggregation
    if 'aggregation' in result:
        print(f'Aggregation done! BACC: {result["aggregation"]["bacc"]*100:.2f}%')
        current_round = result['new_round']
    else:
        current_round = wait_for_aggregation(current_round)

    rounds_completed += 1
    print(f'Progress: {rounds_completed}/{MAX_ROUNDS} rounds')

print('\n' + '='*50)
print(f'FL CLIENT {CLIENT_ID} COMPLETED!')
print(f'Rounds completed: {rounds_completed}')
print('='*50)

Registered! Current round: 0
FL CLIENT PathGen STARTING
Max rounds: 3

--- Round 0 ---
Global model downloaded!
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 145MB/s]


Loaded pretrained weights for efficientnet-b0
Starting local training (1 epochs)...
  Batch 0, Loss: 0.7053
  Batch 10, Loss: 0.5410
Epoch 1: Loss=0.5767, Acc=70.08%
Local training completed!
Uploading weights...
  Original: 26.83 MB
  Compressed: 24.67 MB
  Uploading 17 chunks...
  Chunk 1/17 uploaded
  Chunk 2/17 uploaded
  Chunk 3/17 uploaded
  Chunk 4/17 uploaded
  Chunk 5/17 uploaded
  Chunk 6/17 uploaded
  Chunk 7/17 uploaded
  Chunk 8/17 uploaded
  Chunk 9/17 uploaded
  Chunk 10/17 uploaded
  Chunk 11/17 uploaded
  Chunk 12/17 uploaded
  Chunk 13/17 uploaded
  Chunk 14/17 uploaded
  Chunk 15/17 uploaded
  Chunk 16/17 uploaded
  Chunk 17/17 uploaded
  Upload complete!
Upload result: received
Aggregation done! BACC: 34.00%
Progress: 1/3 rounds

--- Round 1 ---
Global model downloaded!
Loaded pretrained weights for efficientnet-b0
Starting local training (1 epochs)...
  Batch 0, Loss: 0.5133
  Batch 10, Loss: 0.5361
Epoch 1: Loss=0.4769, Acc=76.97%
Local training completed!
Uploadi